 - Task 1:
    - Implement a fully convolutional DCGAN-like model (https://arxiv.org/abs/1511.06434)
    - Train the model on the AFHQ (Animal Faces-HQ) dataset from Assignment 5 in order to generate new animal faces
    - Requirements:
      - Use Tensorboard, WandDB or some other experiment tracker
      - Show the capabilities of your model to generate images
      - Evaluate and track during training using one quantitative metric (e.g. FID)
      - Compare your GAN with your best VAE from Assignment 4.
          - Which model has best FID scores?
          - Which model generates more realistic images?
          - What are the strengths and weaknesses of each model?

In [1]:
import os
import shutil
from tqdm import tqdm
import numpy as np
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
from pytorch_lightning import seed_everything
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import datasets, models, transforms
from torchvision.utils import save_image
from torch.utils.data import DataLoader
from utils import *
import models
# from models import Generator, Discriminator, Trainer ---> imports statically, not compatibel with %load_ext autoreload
from torch.utils.tensorboard import SummaryWriter

/home/user/soltania1/.local/lib/python3.8/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/home/user/soltania1/.local/lib/python3.8/site-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
2025-06-16 23:18:34.155670: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-06-16 23:18:58.301604: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [20]:
%load_ext autoreload
%autoreload 2

In [2]:
configs = {   
    "model_name" : "DCGAN",
    "exp" : "1",  
    "latent_dim" : 128,
    "batch_size" : 64,
    "num_epochs" : 50,
    "lr" : 1e-3,
    "scheduler" : "ReduceLROnPlateau",
    "use_scheduler" : True,
    }

In [3]:
dataset_root = '../Assignment4/data/AFHQ/'

transform = transforms.Compose([transforms.Resize((64,64)),
                                      transforms.ToTensor(),
                                      transforms.Normalize([0.5]*3 , [0.5]*3)])

BS = configs["batch_size"]
latent_dim = configs["latent_dim"]

train_dataset = datasets.ImageFolder(root= dataset_root+'train', transform= transform )
test_dataset = datasets.ImageFolder(root= dataset_root+'test', transform= transform )

# print(train_dataset.classes)  
print(train_dataset.class_to_idx)  

train_loader = DataLoader(dataset= train_dataset, 
                          batch_size= BS, 
                          shuffle= True, 
                          drop_last= True )

test_loader = DataLoader(dataset= test_dataset, 
                          batch_size= BS, 
                          shuffle= False, 
                          drop_last= True )

{'cat': 0, 'dog': 1, 'wild': 2}


In [4]:
generator = models.Generator(latent_dim=latent_dim, num_channels=3, base_channels=64)
print(generator)

Generator(
  (model): Sequential(
    (0): ConvTransposeBlock(
      (block): Sequential(
        (0): ConvTranspose2d(128, 1024, kernel_size=(4, 4), stride=(1, 1), padding=(1, 1))
        (1): BatchNorm2d(1024, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU()
      )
    )
    (1): ConvTransposeBlock(
      (block): Sequential(
        (0): ConvTranspose2d(1024, 512, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
        (1): BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU()
      )
    )
    (2): ConvTransposeBlock(
      (block): Sequential(
        (0): ConvTranspose2d(512, 256, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
        (1): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU()
      )
    )
    (3): ConvTransposeBlock(
      (block): Sequential(
        (0): ConvTranspose2d(256, 128, kernel_size=(4, 4), stride=(2, 2), padding=(1,

In [5]:
gen_img = generator(torch.rand(BS, 128, 1, 1))
print(f'output shape: {gen_img.shape}')

assert gen_img.shape == (BS, 3, 64, 64), "Generator output shape is incorrect! The Generator should output a fake image equal to the size of the training images"

output shape: torch.Size([64, 3, 64, 64])


In [6]:
discriminator = models.Discriminator(in_channels=3, out_dim=1, base_channels=64)
print(discriminator)

Discriminator(
  (model): Sequential(
    (0): ConvBlock(
      (block): Sequential(
        (0): Conv2d(3, 64, kernel_size=(4, 4), stride=(2, 2), padding=(2, 2))
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): LeakyReLU(negative_slope=0.2)
        (3): Dropout(p=0.3, inplace=False)
      )
    )
    (1): ConvBlock(
      (block): Sequential(
        (0): Conv2d(64, 128, kernel_size=(4, 4), stride=(2, 2), padding=(2, 2))
        (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): LeakyReLU(negative_slope=0.2)
        (3): Dropout(p=0.3, inplace=False)
      )
    )
    (2): ConvBlock(
      (block): Sequential(
        (0): Conv2d(128, 256, kernel_size=(4, 4), stride=(2, 2), padding=(2, 2))
        (1): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): LeakyReLU(negative_slope=0.2)
        (3): Dropout(p=0.3, inplace=False)
      )
    )
 

In [7]:
desc_input = torch.rand(BS, 3, 64, 64)
desc_output = discriminator(desc_input)
print(f'output shape: {desc_output.shape}')
assert desc_output.shape == (BS, 1, 1, 1), "Discriminator output shape is incorrect! The Discriminator should output a single value"


output shape: torch.Size([64, 1, 1, 1])


In [8]:
count_model_params(discriminator)

3809857

In [9]:
count_model_params(generator)


13247299

In [22]:
latent_dim = 128
model_name = "DCGAN1"
save_dir = os.path.join(os.getcwd(), "models", model_name, "checkpoint_DCGAN1_epoch_50.pth")

TBOARD_LOGS = os.path.join(os.getcwd(), "tboard_logs", model_name)
writer = SummaryWriter(TBOARD_LOGS)

generator = models.Generator(latent_dim=latent_dim, num_channels=3, base_channels=64)
discriminator = models.Discriminator(in_channels=3, out_dim=1, base_channels=64)

model = models.Trainer(generator=generator, discriminator=discriminator, latent_dim=latent_dim, writer=writer)

def load_model(model, generator_optimizer, discriminator_optimizer, savepath):
    """ Loading pretrained checkpoint """
    
    checkpoint = torch.load(savepath)
    model.generator.load_state_dict(checkpoint['generator_state_dict'])
    model.discriminator.load_state_dict(checkpoint['discriminator_state_dict'])
    generator_optimizer.load_state_dict(checkpoint['optimizer_state_dict_generator'])
    discriminator_optimizer.load_state_dict(checkpoint['optimizer_state_dict_discriminator'])
    epoch = checkpoint["epoch"]
    stats = checkpoint["stats"]
    
    return model, generator_optimizer, discriminator_optimizer, epoch, stats

load_model(model, model.optim_generator, model.optim_discriminator, save_dir)



/tmp/ipykernel_3758135/3613155461.py:16: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(savepath)


(<models.Trainer at 0x7fe795e7a280>,
 Adam (
 Parameter Group 0
     amsgrad: False
     betas: (0.5, 0.9)
     capturable: False
     differentiable: False
     eps: 1e-08
     foreach: None
     fused: None
     lr: 0.0003
     maximize: False
     weight_decay: 0
 ),
 Adam (
 Parameter Group 0
     amsgrad: False
     betas: (0.5, 0.9)
     capturable: False
     differentiable: False
     eps: 1e-08
     foreach: None
     fused: None
     lr: 0.0003
     maximize: False
     weight_decay: 0
 ),
 50,
 {'model_name': 'DCGAN',
  'exp': '1',
  'latent_dim': 128,
  'batch_size': 64,
  'num_epochs': 50,
  'lr': 0.001,
  'scheduler': 'ReduceLROnPlateau',
  'use_scheduler': True})

## GIF 

In [10]:
import imageio
import os
from PIL import Image


images = []
img_dir = os.path.join(os.getcwd(), "imgs", "CDCGAN2")

extensions = ('.png', '.jpg', '.jpeg', '.bmp')
files = sorted([f for f in os.listdir(img_dir) if f.lower().endswith(extensions)])

files = files[0:20000:100]

images = [Image.open(os.path.join(img_dir, f)) for f in files]

images = [img.convert("RGBA") for img in images]

output_path = "CDCGAN.gif"
# Save as GIF
images[0].save(
    output_path,
    save_all=True,
    append_images=images[1:],
    duration=1000,
    loop=0,
    optimize=True
)